<a href="https://colab.research.google.com/github/smartanilmali234-art/Anilmali/blob/main/Week4_ML_Pipeline_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Pipeline & Feature Engineering
**Dataset:** Titanic Survivorship  
**Goal:** Build a modular, leak-free Scikit-Learn `Pipeline` incorporating custom feature engineering, preprocessing with `ColumnTransformer`, and model evaluation.

In [1]:
# System & Data Processing
import numpy as np
import pandas as pd

# Model Selection & Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Preprocessing & Pipeline Components
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Estimators & Serialization
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import joblib

print("All dependencies successfully imported.")

All dependencies successfully imported.


## 1. Data Ingestion
We load the raw Titanic dataset using `seaborn` (or OpenML) to simulate raw input data entering our ML workflow.

In [2]:
import seaborn as sns

# Load dataset
df = sns.load_dataset('titanic')

# Standardize column naming for consistency
df = df.rename(columns={'sex': 'Sex', 'age': 'Age', 'fare': 'Fare',
                        'sibsp': 'SibSp', 'parch': 'Parch',
                        'embarked': 'Embarked', 'pclass': 'Pclass',
                        'survived': 'Survived'})

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Exploratory Data Inspection
Let's examine dataset dimensions, data types, and missing value counts.

In [3]:
print("Data Shape:", df.shape)
print("\nMissing Values per Feature:")
print(df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].isnull().sum())

Data Shape: (891, 15)

Missing Values per Feature:
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64


## 3. Train-Test Split
**Crucial Step:** We split raw data into training and test sets *before* performing any transformation or imputation to prevent **Data Leakage**.

In [4]:
# Define features and target variable
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features]
y = df['Survived']

# Stratified split to preserve target ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Samples: {X_train.shape[0]} | Test Samples: {X_test.shape[0]}")

Training Samples: 712 | Test Samples: 179


## 4. Manual / Unpipelined Approach (Baseline)
To quantify the benefit of Scikit-Learn Pipelines, we first preprocess the data manually using standard pandas/sklearn calls.

In [5]:
# Copy datasets to avoid mutating originals
X_tr_manual = X_train.copy()
X_te_manual = X_test.copy()

# Manual Imputation
X_tr_manual['Age'] = X_tr_manual['Age'].fillna(X_tr_manual['Age'].median())
X_te_manual['Age'] = X_te_manual['Age'].fillna(X_tr_manual['Age'].median())

X_tr_manual['Embarked'] = X_tr_manual['Embarked'].fillna(X_tr_manual['Embarked'].mode()[0])
X_te_manual['Embarked'] = X_te_manual['Embarked'].fillna(X_tr_manual['Embarked'].mode()[0])

# Manual One-Hot Encoding & Scaling
X_tr_encoded = pd.get_dummies(X_tr_manual, columns=['Sex', 'Embarked'], drop_first=True)
X_te_encoded = pd.get_dummies(X_te_manual, columns=['Sex', 'Embarked'], drop_first=True)

# Align columns
X_tr_encoded, X_te_encoded = X_tr_encoded.align(X_te_encoded, join='left', axis=1, fill_value=0)

# Manual Scaling
scaler = StandardScaler()
num_cols = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']
X_tr_encoded[num_cols] = scaler.fit_transform(X_tr_encoded[num_cols])
X_te_encoded[num_cols] = scaler.transform(X_te_encoded[num_cols])

### Fitting and Evaluating Manual Baseline Model
We fit a basic Random Forest Classifier on our manually prepared features.

In [6]:
manual_model = RandomForestClassifier(random_state=42)
manual_model.fit(X_tr_encoded, y_train)

y_pred_manual = manual_model.predict(X_te_encoded)
acc_manual = accuracy_score(y_test, y_pred_manual)
f1_manual = f1_score(y_test, y_pred_manual)

print(f"Manual Baseline Accuracy: {acc_manual:.4f}")
print(f"Manual Baseline F1-Score: {f1_manual:.4f}")

Manual Baseline Accuracy: 0.8101
Manual Baseline F1-Score: 0.7463


## 5. Engineering Custom Features (Scikit-Learn Compatible)
We create a custom Scikit-Learn transformer class by inheriting from `BaseEstimator` and `TransformerMixin`.

**Engineered Features:**
1. `family_size = SibSp + Parch + 1`
2. `is_alone = 1 if family_size == 1 else 0`

In [7]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, drop_original=False):
        self.drop_original = drop_original

    def fit(self, X, y=None):
        return self  # No parameters to estimate

    def transform(self, X):
        X_out = X.copy()

        # Feature 1: Family Size
        X_out['family_size'] = X_out['SibSp'] + X_out['Parch'] + 1

        # Feature 2: Is Alone Flag
        X_out['is_alone'] = (X_out['family_size'] == 1).astype(int)

        if self.drop_original:
            X_out = X_out.drop(columns=['SibSp', 'Parch'])

        return X_out

### Testing the Custom Feature Engineer Transformer
Let's verify that our custom transformer works smoothly on a small sample dataframe.

In [8]:
fe_transformer = TitanicFeatureEngineer()
sample_transformed = fe_transformer.transform(X_train.head(3))
sample_transformed[['SibSp', 'Parch', 'family_size', 'is_alone']]

,SibSp,Parch,family_size,is_alone
692,0,0,1,1
481,0,0,1,1
527,0,0,1,1


## 6. Building Column Preprocessing Sub-Pipelines
We specify separate pipelines for numerical features (Imputation + Scaling) and categorical features (Imputation + OneHotEncoding).

In [9]:
# Identify numerical and categorical features
num_features = ['Age', 'Fare', 'Pclass', 'family_size', 'is_alone', 'SibSp', 'Parch']
cat_features = ['Sex', 'Embarked']

# Sub-pipeline for numerical columns
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Sub-pipeline for categorical columns
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

## 7. Combining Transformations with ColumnTransformer
`ColumnTransformer` routes each set of features to its respective sub-pipeline.

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ]
)

## 8. Assembling the Full ML Pipeline
We chain Feature Engineering, Column Preprocessing, and Model Estimator into a single reusable object.

In [11]:
full_pipeline = Pipeline([
    ('feature_engineer', TitanicFeatureEngineer(drop_original=False)),
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Inspect pipeline architecture
full_pipeline

Pipeline(steps=[('feature_engineer', TitanicFeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'Pclass',
                                                   'family_size', 'is_alone',
                                                   'SibSp', 'Parch']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

## 9. Fitting the Full ML Pipeline
Calling `.fit()` triggers the sequence: transform training features, apply column preprocessing, and train the classifier without any data leakage.

In [12]:
full_pipeline.fit(X_train, y_train)
print("Pipeline successfully fitted on training data!")

Pipeline successfully fitted on training data!


## 10. Model Evaluation & Comparison
We evaluate predictions on the unseen holdout test set (`X_test`).

In [13]:
# Generate predictions directly from raw test data
y_pred_pipe = full_pipeline.predict(X_test)

acc_pipe = accuracy_score(y_test, y_pred_pipe)
f1_pipe = f1_score(y_test, y_pred_pipe)

print("=== PIPELINE RESULTS ===")
print(f"Pipeline Test Accuracy : {acc_pipe:.4f}")
print(f"Pipeline Test F1-Score : {f1_pipe:.4f}")

print("\n=== PERFORMANCE COMPARISON ===")
comparison_df = pd.DataFrame({
    'Approach': ['Manual Baseline', 'Scikit-Learn Pipeline'],
    'Accuracy': [acc_manual, acc_pipe],
    'F1-Score': [f1_manual, f1_pipe]
})
print(comparison_df.to_string(index=False))

=== PIPELINE RESULTS ===
Pipeline Test Accuracy : 0.8101
Pipeline Test F1-Score : 0.7424

=== PERFORMANCE COMPARISON ===
             Approach  Accuracy  F1-Score
      Manual Baseline  0.810056  0.746269
Scikit-Learn Pipeline  0.810056  0.742424


## 11. Pipeline Serialization (Saving the Pipeline)
We export the entire end-to-end pipeline using `joblib` so it can be deployed directly into production environments.

In [14]:
model_filename = 'titanic_ml_pipeline.joblib'

# Save full pipeline to disk
joblib.dump(full_pipeline, model_filename)
print(f"Pipeline successfully saved to {model_filename}")

Pipeline successfully saved to titanic_ml_pipeline.joblib


## 12. Model Deserialization & Inference Verification
We reload the stored pipeline and test inference on new raw passenger records.

In [15]:
# Load saved model back into memory
loaded_pipeline = joblib.load(model_filename)

# Define mock raw passenger data (unprocessed)
new_passengers = pd.DataFrame([
    {
        'Pclass': 3,
        'Sex': 'male',
        'Age': 22.0,
        'SibSp': 1,
        'Parch': 0,
        'Fare': 7.25,
        'Embarked': 'S'
    },
    {
        'Pclass': 1,
        'Sex': 'female',
        'Age': 38.0,
        'SibSp': 1,
        'Parch': 0,
        'Fare': 71.28,
        'Embarked': 'C'
    }
])

# Predict survival status
predictions = loaded_pipeline.predict(new_passengers)
probabilities = loaded_pipeline.predict_proba(new_passengers)[:, 1]

for i, pred in enumerate(predictions):
    status = "Survived" if pred == 1 else "Did Not Survive"
    print(f"Passenger {i+1}: Prediction = {status} | Survival Prob = {probabilities[i]:.2%}")

Passenger 1: Prediction = Did Not Survive | Survival Prob = 11.00%
Passenger 2: Prediction = Survived | Survival Prob = 100.00%


## 13. Summary & Insights
1. **Zero Data Leakage:** Imputation statistics and scaling factors were calculated strictly on training folds.
2. **Reproducibility & Cleaner Code:** Replaced dozens of lines of pandas manipulation with standard `.fit()` and `.predict()` calls.
3. **Engineered Features:** `family_size` and `is_alone` provided additional signals for the decision tree split.

In [16]:
# Final confirmation log
print("Task Complete: Machine Learning Pipeline built, evaluated, and saved successfully.")

Task Complete: Machine Learning Pipeline built, evaluated, and saved successfully.
